In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 1. Carregar os resultados do modelo

In [ ]:
import pandas as pd

caminho_arquivo = '/caminho/para/geracoes.csv'

df = pd.read_csv(caminho_arquivo)
df.head()

## 2. Carregar o dataset de referência (APPS Benchmark)

In [ ]:
from datasets import load_dataset

dataset = load_dataset(
    "json",
    data_files="hf://datasets/codeparrot/apps/test.jsonl",
    split="train"
)

## 3. Funções de extração e avaliação de código

Extrai o código Python de uma resposta.

In [ ]:
import subprocess
import sys
import re
import json


def extrair_codigo(texto: str) -> str:
    texto = texto.strip()

    # (```python ... ```)
    match = re.search(r"```python\s*\n(.*?)\n```", texto, re.DOTALL)
    if match:
        return match.group(1).strip()

    # (``` ... ```)
    match = re.search(r"```\s*\n(.*?)\n```", texto, re.DOTALL)
    if match:
        return match.group(1).strip()

    return texto


def avaliar_codigo(resposta: str, testes_dict: dict, timeout_segundos: int = 2) -> tuple[int, int]:
    # Executa o código para cada caso de teste e retorna (acertos, total).
    codigo = extrair_codigo(resposta)

    entradas = testes_dict.get("inputs", [])
    saidas_esperadas = testes_dict.get("outputs", [])
    total = len(entradas)
    testes_passados = 0

    for entrada, saida_esperada in zip(entradas, saidas_esperadas):
        try:
            processo = subprocess.run(
                [sys.executable, "-c", codigo],
                input=str(entrada),
                text=True,
                capture_output=True,
                timeout=timeout_segundos
            )

            if processo.returncode == 0:
                saida_obtida = processo.stdout.strip()
                saida_correta = str(saida_esperada).strip()

                if saida_obtida == saida_correta:
                    testes_passados += 1
        except subprocess.TimeoutExpired:
            pass
        except Exception:
            pass

    return testes_passados, total

## 4. Avaliação em massa

Roda a função ```avaliar_codigo()``` para todas as respostas geradas o avalia para os casos de teste presente no dataset.

In [ ]:
import csv
import os
import shutil
from tqdm.notebook import tqdm

ARQUIVO_LOCAL = 'avaliacao.csv'

PASTA_DRIVE = '/content/drive/MyDrive/resultados_eval_llm/'
ARQUIVO_DRIVE = os.path.join(PASTA_DRIVE, 'avaliacao.csv')
os.makedirs(PASTA_DRIVE, exist_ok=True)
INTERVALO_CHECKPOINT = 25

def salvar_checkpoint_drive():
    if os.path.exists(ARQUIVO_LOCAL):
        shutil.copyfile(ARQUIVO_LOCAL, ARQUIVO_DRIVE)

ja_avaliados = set()
if os.path.exists(ARQUIVO_DRIVE):
    shutil.copyfile(ARQUIVO_DRIVE, ARQUIVO_LOCAL)

if os.path.exists(ARQUIVO_LOCAL):
    df_existente = pd.read_csv(ARQUIVO_LOCAL)
    ja_avaliados = set(df_existente['question_id'].tolist())

escrever_cabecalho = not os.path.exists(ARQUIVO_LOCAL)

with open(ARQUIVO_LOCAL, 'a', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=['question_id', 'acertos', 'total', 'testes'])
    if escrever_cabecalho:
        writer.writeheader()

    linhas_pendentes = [l for l in df.itertuples(index=False) if l.question_id not in ja_avaliados]

    for i, linha in enumerate(tqdm(linhas_pendentes, desc="Avaliando questões"), start=1):
        question_id = linha.question_id

        try:
            questao = dataset[question_id]
            dados_testes = json.loads(questao.get("input_output"))
        except (IndexError, KeyError, TypeError, json.JSONDecodeError):
            writer.writerow({
                'question_id': question_id,
                'acertos': 0,
                'total': 0,
                'testes': "0/0",
            })
            f.flush()
            continue

        acertos, total = avaliar_codigo(linha.response, dados_testes)

        writer.writerow({
            'question_id': question_id,
            'acertos': acertos,
            'total': total,
            'testes': f"{acertos}/{total}",
        })
        f.flush()

        if i % INTERVALO_CHECKPOINT == 0:
            salvar_checkpoint_drive()

salvar_checkpoint_drive()
print(f"Checkpoint final salvo em: {ARQUIVO_DRIVE}")

## 5. Resultados por questão

In [ ]:
df_avaliacao = pd.read_csv(ARQUIVO_DRIVE)
df_avaliacao.head()

## 6. Taxa de acerto geral

In [ ]:
acertos_totais = df_avaliacao['acertos'].sum()
testes_totais = df_avaliacao['total'].sum()

print(f"{acertos_totais}/{testes_totais}")
print(f"Porcentagem de acertos: {acertos_totais / testes_totais * 100:.2f}%")

## 7. Taxa de acerto por dificuldade

In [ ]:
df_completo = df_avaliacao.merge(df[['question_id', 'difficulty']], on='question_id', how='left')

resumo_dificuldade = (
    df_completo
    .groupby('difficulty')[['acertos', 'total']]
    .sum()
)
resumo_dificuldade['porcentagem'] = (
    resumo_dificuldade['acertos'] / resumo_dificuldade['total'] * 100
).round(2)

resumo_dificuldade

## 8. Gráfico da taxa de acerto por dificuldade

In [ ]:
import matplotlib.pyplot as plt

resumo_dificuldade['porcentagem'].plot(kind='bar', figsize=(6, 4))
plt.ylabel('% de testes corretos')
plt.title('Taxa de acerto por nível de dificuldade')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 9. Questões com 0% de acerto

Útil para inspecionar manualmente os casos em que o modelo errou tudo — pode indicar bug na extração do código, timeout, ou um problema realmente difícil.

In [ ]:
questoes_zeradas = df_completo[(df_completo['total'] > 0) & (df_completo['acertos'] == 0)]
print(f"{len(questoes_zeradas)} questões com 0% de acerto de um total de {len(df_completo)}")
questoes_zeradas.head(20)

## Encerramento do ambiente de execução
Desconecta a sessão do Google Colabs para economizar recursos computacionais.

In [ ]:
from google.colab import runtime
runtime.unassign()